# Bebras Task and API access
This notebook is meant to access API of various LLMs (ChatGPT 4o, Mistal AI, Gemini 2.5, Claude Sonnet 4, DeepSeek). <br>
Before doing so, API access was tested and additional scripts written. 

**Download necessary packages & set paths** <br>
Make sure to insert the key below before running anything. For security reasons, they are not written in this notebook

In [ ]:
claude_key = ""
mistral_key = ""
gemini_key = ""
gpt_key = ""
deepseek_key = ""

In [55]:
#Package for Claude (anthropic)
#!pip install anthropic
import anthropic

#Package for Gemini
#!pip install google-generativeai
import google.generativeai as genai

# For GPT 
from openai import OpenAI

#Package for Mistral AI
#!pip install mistralai
#!pip install "mistralai[agents]"

#Package for DeepSeek
#!pip install openai

from pathlib import Path
import os
from collections import defaultdict
import json 
import time
from datetime import datetime
import openai
from pathlib import Path



in_test_path = r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\TestAPI\Test.txt"
#Access our task directionary
root_path = Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks\input")
output_path = Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks\output")

In [ ]:
#Access (Claude Sonnet 4) - THIS DOESN'T WORK YET

client = anthropic.Anthropic(api_key="INSERTKEYHERE")
#Unfortunately, I have to pay for that one :( 
client.beta.files.upload(
  file=("Test.txt", open(in_test_path, "rb"), "text/plain"),
)

In [ ]:
# TEST: Access Gemini - Test works!

genai.configure(api_key = gemini_key)

model = genai.GenerativeModel("gemini-2.5-flash")

#response = model.generate_content("Explain how AI works in a few words")
with open(in_test_path, "r", encoding="utf-8") as file:
    prompt = file.read()
response = model.generate_content(f"{prompt}") 

test_output_path = r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\TestAPI\test_gemini_2,5.txt"
with open(test_output_path, "w", encoding="utf-8") as f:
    f.write(response.text)

print(response.text)

In [ ]:
#TEST: Access Mistral AI - Test works!
os.environ["MISTRAL_API_KEY"] = mistral_key

from mistralai import Mistral
import os

# Initialize client
client = Mistral(api_key= mistral_key)

with open(in_test_path, "r", encoding="utf-8") as file:
    prompt = file.read()

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user",
               "content": f"{prompt}"}]
)

output_path = r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\TestAPI\test_mistralai.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(response.choices[0].message.content)

print(response.choices[0].message.content)


In [ ]:
#TEST: Access DeepSeek - PAYEMENT REQUIRED

from openai import OpenAI

client = OpenAI(api_key= deepseek_key, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user",
         "content": "What is the time in Japan right now?"}
    ],
    stream=False
)

print(response.choices[0].message.content)


In [ ]:
#TEST: Access GPT - NOT YET

from openai import OpenAI

client = OpenAI(
  api_key= gpt_key)

completion = client.chat.completions.create(
  model="gpt-4.1",
  store=True,
  messages=[
    {"role": "user", "content": "This is a test. Respond with 'yes' if you get this"}
  ]
)

print(completion.choices[0].message)


ChatCompletionMessage(content='Lines of code entwined,  \nSilent thoughts in circuits hum,  \nDreams born from machines.  ', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


**Access to folder-paths** <br>
Now, we want to submit the tasks. Therefore, we first need a script which runs through each folder.

In [5]:
# I decided to save the paths in a directionary so that it's ordered neatly

from pathlib import Path
import os
from collections import defaultdict

age_groups = ["year_7_8", "year_9_10", "year_11_12"]
difficulties = ["easy", "medium", "hard"]
tasks = defaultdict(dict)

for age in age_groups:
    for dif in difficulties:
        tasks[age][dif] = []
    
def scandir_recursive(directory, age, difficulty):
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file():
                tasks[age][difficulty].append(entry.path)
                #print(entry.path)
            elif entry.is_dir():
                #print(entry.name)
                if entry.name in age_groups:
                    cur_age = entry.name
                    scandir_recursive(entry.path, cur_age, difficulty)
                elif entry.name in difficulties:
                    cur_difficulty = entry.name  
                    scandir_recursive(entry.path, age, cur_difficulty)
            
scandir_recursive(root_path,"","")

#Test to see whether the paths were saved - They were!
#print(tasks["year_7_8"])
#print(tasks["year_9_10"]["easy"])


**Answers** <br>
We want to make sure, models are re-prompted if the answer is incorrect. <br>
Therefore, we have to save the correct answer. 

In [6]:
answers = defaultdict(lambda: defaultdict(dict))
#Save the names of each task
tname_7_8_easy = ["BeckyBee", "FlowerGarden", "StoneFactory", "Strawberries","TheGift"]
tname_7_8_medium = ["ClassroomSeating", "ColourTheFrog!", "OhridPearls", "Tic-Tac-Toe","WheatStorage"]
tname_7_8_hard = ["ConnectionOfIslands", "FavouriteMovie", "Mysteria", "Ordering1","UndergroundTrainNetwork"]

tname_9_10_easy = ["Cipher8", "InLove", "MarysNeighbours", "NutsAndBolts","RugWeaving"]
tname_9_10_medium = ["BeaverAI", "HangarCarousel", "OverlappingVillages", "SuperSecuritySystem","ThePrinter"]
tname_9_10_hard = ["CollectingStones", "FavouriteGem", "FourTiles", "Maze","TreeFarming"]

tname_11_12_easy = ["ColourfulCandles", "ListenAndWalk", "Lists", "MovieNight","Words"]
tname_11_12_medium = ["BeaverDam", "JumpingGame", "Packing", "TreasureBox"]
tname_11_12_hard = ["BeaverDatabase", "BeaverGames", "Ordering2", "SeashellsAndPebbles","Virus"]

tasks_by_group = {
    ("year_7_8", "easy"): tname_7_8_easy,
    ("year_7_8", "medium"): tname_7_8_medium,
    ("year_7_8", "hard"): tname_7_8_hard,
    ("year_9_10", "easy"): tname_9_10_easy,
    ("year_9_10", "medium"): tname_9_10_medium,
    ("year_9_10", "hard"): tname_9_10_hard,
    ("year_11_12", "easy"): tname_11_12_easy,
    ("year_11_12", "medium"): tname_11_12_medium,
    ("year_11_12", "hard"): tname_11_12_hard,
}

# To make it easier to access, I will save it just like the task paths
for (age, dif), task_list in tasks_by_group.items():
    for task in task_list:
        answers[age][dif][task] = []

In [39]:
# Insert answers
# IMPORTANT: A few tasks do not require a multiple chioce answer,
# which is why I will just re-prompt them regardless and check the repsonse manualy

# **Year 7/8**
# Easy
answers["year_7_8"]["easy"]["BeckyBee"] = "5-10-15-7-10-5-5-10-5"
answers["year_7_8"]["easy"]["FlowerGarden"] = "B"
answers["year_7_8"]["easy"]["StoneFactory"] = "Output 2"
answers["year_7_8"]["easy"]["Strawberries"] = "23"
answers["year_7_8"]["easy"]["TheGift"] = "Box 4"

# Medium
answers["year_7_8"]["medium"]["ClassroomSeating"] = "Statement 3"
answers["year_7_8"]["medium"]["ColourTheFrog!"] = "Block 4"
answers["year_7_8"]["medium"]["OhridPearls"] = "Option 1"
answers["year_7_8"]["medium"]["Tic-Tac-Toe"] = "Option 3"
answers["year_7_8"]["medium"]["WheatStorage"] = "50 minutes"

# Hard
answers["year_7_8"]["hard"]["FavouriteMovie"] = "Niklaus and Rozsa"
answers["year_7_8"]["hard"]["Mysteria"] = "Option B"
answers["year_7_8"]["hard"]["ConnectionOfIslands"] = "44"
answers["year_7_8"]["hard"]["Ordering1"] = "415236"
answers["year_7_8"]["hard"]["UndergroundTrainNetwork"] = "Station B"

# **Year 9/10**
# Easy
answers["year_9_10"]["easy"]["NutsAndBolts"] = "Option 3"
answers["year_9_10"]["easy"]["MarysNeighbours"] = "House 4"
answers["year_9_10"]["easy"]["Cipher8"] = "72-11-62-32-43"
answers["year_9_10"]["easy"]["RugWeaving"] = "Option 2"
answers["year_9_10"]["easy"]["InLove"] = "Robert"

# Medium
# Manual
answers["year_9_10"]["medium"]["SuperSecuritySystem"] = "NA"  
answers["year_9_10"]["medium"]["HangarCarousel"] = "4-1-3-6-2-5 or 4-1-5-2-6-3"
answers["year_9_10"]["medium"]["OverlappingVillages"] = "5"
answers["year_9_10"]["medium"]["BeaverAI"] = "Option 3"
answers["year_9_10"]["medium"]["ThePrinter"] = "11-21-12-22-32-31-13-23-33 at 10:22"

# Hard
answers["year_9_10"]["hard"]["FourTiles"] = "Tile 3"
answers["year_9_10"]["hard"]["FavouriteGem"] = "10"
answers["year_9_10"]["hard"]["CollectingStones"] = "14 beavers"
answers["year_9_10"]["hard"]["Maze"] = "18"
#Manual
answers["year_9_10"]["hard"]["TreeFarming"] = "NA"  

# **Year 11/12**
# Easy
answers["year_11_12"]["easy"]["Lists"] = "4"
answers["year_11_12"]["easy"]["ListenAndWalk"] = "B"
answers["year_11_12"]["easy"]["ColourfulCandles"] = "5"
answers["year_11_12"]["easy"]["MovieNight"] = "0-Mo, 1-Joud, 2-Sara, 3-Han, 4-Emy, 5-Jon, 6-Mary"
answers["year_11_12"]["easy"]["Words"] = "9"

# Medium
answers["year_11_12"]["medium"]["BeaverDam"] = "Option 4"
answers["year_11_12"]["medium"]["JumpingGame"] = "[Verunka,X][X][O][X][X][O][O][X][X][O][O][X][X][O][O][Coin]"
#Manual
answers["year_11_12"]["medium"]["Packing"] = "NA"
answers["year_11_12"]["medium"]["TreasureBox"] = "Option 2"

# Hard
answers["year_11_12"]["hard"]["BeaverGames"] = "D-F-G"
answers["year_11_12"]["hard"]["BeaverDatabase"] = "Option 3"
answers["year_11_12"]["hard"]["Ordering2"] = "1-2-3-4-6-5-B-A-7-8-9"
answers["year_11_12"]["hard"]["Virus"] = "6"
answers["year_11_12"]["hard"]["SeashellsAndPebbles"] = "Hole 7"



# API Calls + tasks - Baseline
This is where the fun begins. Our task amount is not big enough for "batching", so this won't be happening.<br>
Unfrotunately, because models differ in their API calling, each model will have a seperate function which will be called in its own code block. <br>
An output folder has already been created. 

**Gemini 2.5**

In [ ]:
# Let's test first with one task - the end function is now very different
genai.configure(api_key = gemini_key)

model = genai.GenerativeModel("gemini-2.5-flash")

gemini_test_path = tasks["year_7_8"]["easy"][1]
#response = model.generate_content("Explain how AI works in a few words")
with open(gemini_test_path, "r", encoding="utf-8") as file:
    prompt = file.read()
response = model.generate_content("Consider the following task. Please think about it carefully and provide an explanation for your answer."f"{prompt}") 

test_output_path = output_path / "year_7_8" / "easy" / "test_Gemini_2,5.txt"

with open(test_output_path, "w", encoding="utf-8") as f:
    f.write(response.text)

print(response.text)

In [35]:
# Base function for gemini
# We want the model to give us an explanation on how it came to its answer
# If the answer does not match up with the correct answer, we need to reprompt

# Load API key and model
genai.configure(api_key=gemini_key)

def gemini(age, dif):
    model = genai.GenerativeModel("gemini-2.5-flash")
    task_path = tasks[age][dif]

    # Define schema for structured output
    function_schema = {
        "type": "object",
        "properties": {
            "explanation": {"type": "string", "description": "Explanation of the reasoning process"},
            "answer": {"type": "string", "description": "Final answer"}
        },
        "required": ["explanation", "answer"]
    }

    for task in task_path:
        task_name = os.path.splitext(os.path.basename(task))[0]
        with open(task, "r", encoding="utf-8") as file:
            prompt = file.read()

        print(f"Generating response for: {task_name}")

        # Construct input prompt
        full_prompt = (
            "Consider the following task. Please think about it carefully and provide: \n"
            "1. 'explanation': How you solved the task.\n"
            "2. 'answer': Only the final answer (e.g. 'option 2', 'block 3', etc. unless specified differntly). Do not add anything else.\n\n"
            f"{prompt}"
        )

        # Measure time and get response
        start_time = time.time()
        response = model.generate_content(
            full_prompt,
            tools=[{"function_declarations": [{"name": "structured_output", "description": "Outputs the explanation and final answer.", "parameters": function_schema}]}],
            tool_config={"function_calling_config": {"mode": "ANY"}}
        )
        end_time = time.time()
        response_time = round(end_time - start_time, 3)

        # Parse tool response
        tool_result = response.candidates[0].content.parts[0].function_call.args
        explanation = tool_result.get("explanation", "").strip()
        predicted = tool_result.get("answer", "").strip()

        correct_answer = answers[age][dif][task_name].strip()
        is_correct = predicted.lower() == correct_answer.lower()

        # Save structured first output
        output_data = {
            "task_name": task_name,
            "prompt": prompt,
            "explanation": explanation,
            "answer": predicted,
            "model": "gemini-2.5-flash",
            "timestamp": datetime.now().isoformat(),
            "response_time_sec": response_time,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "attempt": "first"
        }

        gemini_output_path = output_path / f"{age}" / f"{dif}" / "gemini"
        gemini_output_path.mkdir(parents=True, exist_ok=True)
        with open(gemini_output_path / f"{task_name}_Gemini_2.5.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4, ensure_ascii=False)

        print(f"Saved first output for: {task_name}")

        # Retry logic if incorrect
        if not is_correct:
            print(f"Wrong answer detected for: {task_name}")
            print(f"Model answered: {predicted}")
            print(f"Correct answer: {correct_answer}")
            print("Re-prompting...")

            retry_prompt = (
                f"You previously answered incorrectly.\n"
                f"Your explanation: {explanation}\n"
                f"Your answer: {predicted}\n\n"
                f"Please rethink the problem and try again.\n\n"
                f"{prompt}"
            )

            start_time = time.time()
            retry_response = model.generate_content(
                retry_prompt,
                tools=[{"function_declarations": [{"name": "structured_output", "description": "Outputs the explanation and final answer of the retry.", "parameters": function_schema}]}],
                tool_config={"function_calling_config": {"mode": "ANY"}}
            )
            end_time = time.time()
            retry_time = round(end_time - start_time, 3)

            retry_result = retry_response.candidates[0].content.parts[0].function_call.args
            retry_expl = retry_result.get("explanation", "").strip()
            retry_ans = retry_result.get("answer", "").strip()

            retry_data = {
                "task_name": task_name,
                "prompt": prompt,
                "previous_answer": predicted,
                "explanation": retry_expl,
                "answer": retry_ans,
                "model": "gemini-2.5-flash",
                "timestamp": datetime.now().isoformat(),
                "response_time_sec": retry_time,
                "attempt": "retry"
            }

            with open(gemini_output_path / f"{task_name}_Gemini_2.5_retry.json", "w", encoding="utf-8") as f:
                json.dump(retry_data, f, indent=4, ensure_ascii=False)

            print(f"Saved retry output for: {task_name}")


**Calling Gemini functions** <br>
Will do each function call seperately. That way, I can look at the results manually and step by step <br>
<br>
**Year 7/8**

In [36]:
# Year 7/8 + Easy
gemini("year_7_8","easy")

Generating response for: BeckyBee
Saved first output for: BeckyBee
Wrong answer detected for: BeckyBee
Model answered: 5-15-7-10-9-20-5-10-5
Correct answer: 5-10-15-7-10-5-5-10-5
Re-prompting...
Saved retry output for: BeckyBee
Generating response for: FlowerGarden
Saved first output for: FlowerGarden
Generating response for: StoneFactory
Saved first output for: StoneFactory
Generating response for: Strawberries
Saved first output for: Strawberries
Wrong answer detected for: Strawberries
Model answered: 28
Correct answer: 23
Re-prompting...
Saved retry output for: Strawberries
Generating response for: TheGift
Saved first output for: TheGift


In [37]:
# Year 7/8 + Medium
gemini("year_7_8","medium")

Generating response for: ClassroomSeating
Saved first output for: ClassroomSeating
Generating response for: ColourTheFrog!
Saved first output for: ColourTheFrog!
Generating response for: OhridPearls
Saved first output for: OhridPearls
Generating response for: Tic-Tac-Toe
Saved first output for: Tic-Tac-Toe
Wrong answer detected for: Tic-Tac-Toe
Model answered: Option 1
Correct answer: option 3
Re-prompting...
Saved retry output for: Tic-Tac-Toe
Generating response for: WheatStorage
Saved first output for: WheatStorage
Wrong answer detected for: WheatStorage
Model answered: 54 minutes
Correct answer: 50 minutes
Re-prompting...
Saved retry output for: WheatStorage


In [40]:
# Year 7/8 + Hard
gemini("year_7_8","hard")

Generating response for: ConnectionOfIslands
Saved first output for: ConnectionOfIslands
Generating response for: FavouriteMovie
Saved first output for: FavouriteMovie
Wrong answer detected for: FavouriteMovie
Model answered: Niklaus, Rozsa
Correct answer: Niklaus and Rozsa
Re-prompting...
Saved retry output for: FavouriteMovie
Generating response for: Mysteria
Saved first output for: Mysteria
Wrong answer detected for: Mysteria
Model answered: Option A
Correct answer: Option B
Re-prompting...
Saved retry output for: Mysteria
Generating response for: Ordering1
Saved first output for: Ordering1
Generating response for: UndergroundTrainNetwork
Saved first output for: UndergroundTrainNetwork


**Year 9/10**

In [41]:
# Year 9/10 + Easy
gemini("year_9_10","easy")

Generating response for: Cipher8
Saved first output for: Cipher8
Generating response for: InLove
Saved first output for: InLove
Generating response for: MarysNeighbours
Saved first output for: MarysNeighbours
Wrong answer detected for: MarysNeighbours
Model answered: House 5
Correct answer: House 4
Re-prompting...
Saved retry output for: MarysNeighbours
Generating response for: NutsAndBolts
Saved first output for: NutsAndBolts
Wrong answer detected for: NutsAndBolts
Model answered: Option 1
Correct answer: Option 3
Re-prompting...
Saved retry output for: NutsAndBolts
Generating response for: RugWeaving
Saved first output for: RugWeaving


In [42]:
# Year 9/10 + Medium
gemini("year_9_10","medium")

Generating response for: BeaverAI
Saved first output for: BeaverAI
Generating response for: HangarCarousel
Saved first output for: HangarCarousel
Wrong answer detected for: HangarCarousel
Model answered: 4-1-3-6-2-5
Correct answer: 4-1-3-6-2-5 or 4-1-5-2-6-3
Re-prompting...
Saved retry output for: HangarCarousel
Generating response for: OverlappingVillages
Saved first output for: OverlappingVillages
Wrong answer detected for: OverlappingVillages
Model answered: 4
Correct answer: 5
Re-prompting...
Saved retry output for: OverlappingVillages
Generating response for: SuperSecuritySystem
Saved first output for: SuperSecuritySystem
Wrong answer detected for: SuperSecuritySystem
Model answered: The five detectors should be placed at the following 0-indexed coordinates (row, column): (0,0), (1,2), (2,4), (3,1), (4,3).
Correct answer: NA
Re-prompting...
Saved retry output for: SuperSecuritySystem
Generating response for: ThePrinter
Saved first output for: ThePrinter


In [43]:
# Year 9/10 + Hard
gemini("year_9_10","hard")

Generating response for: CollectingStones
Saved first output for: CollectingStones
Wrong answer detected for: CollectingStones
Model answered: 16 beavers
Correct answer: 14 beavers
Re-prompting...
Saved retry output for: CollectingStones
Generating response for: FavouriteGem
Saved first output for: FavouriteGem
Generating response for: FourTiles
Saved first output for: FourTiles
Wrong answer detected for: FourTiles
Model answered: Tile 2
Correct answer: Tile 3
Re-prompting...
Saved retry output for: FourTiles
Generating response for: Maze
Saved first output for: Maze
Wrong answer detected for: Maze
Model answered: 17
Correct answer: 18
Re-prompting...
Saved retry output for: Maze
Generating response for: TreeFarming
Saved first output for: TreeFarming
Wrong answer detected for: TreeFarming
Model answered: Trees at (2,2), (4,2), (5,1), (6,1), (6,2), (7,3), (8,2), (9,1)
Correct answer: NA
Re-prompting...
Saved retry output for: TreeFarming


**Year 11/12**

In [45]:
# Year 11/12 + Easy
gemini("year_11_12","easy")

Generating response for: ColourfulCandles
Saved first output for: ColourfulCandles
Generating response for: ListenAndWalk
Saved first output for: ListenAndWalk
Wrong answer detected for: ListenAndWalk
Model answered: C
Correct answer: B
Re-prompting...
Saved retry output for: ListenAndWalk
Generating response for: Lists
Saved first output for: Lists
Generating response for: MovieNight
Saved first output for: MovieNight
Generating response for: Words
Saved first output for: Words


In [46]:
# Year 11/12 + Medium
gemini("year_11_12","medium")

Generating response for: BeaverDam
Saved first output for: BeaverDam
Generating response for: JumpingGame
Saved first output for: JumpingGame
Wrong answer detected for: JumpingGame
Model answered: Verunka,XXOOXXOOXXOOXXOOCoin
Correct answer: [Verunka,X][X][O][X][X][O][O][X][X][O][O][X][X][O][O][Coin]
Re-prompting...
Saved retry output for: JumpingGame
Generating response for: Packing
Saved first output for: Packing
Wrong answer detected for: Packing
Model answered: Place the green bar (5x3) at (0,0). Rotate the violet bar (1x5) to 5x1 and place it at (0,3). Place the pink bar (3x4) at (0,4). Rotate the white bar (3x2) to 2x3 and place it at (3,4). This arrangement creates an 8x5 rectangular gift box with 2 gaps.
Correct answer: NA
Re-prompting...
Saved retry output for: Packing
Generating response for: TreasureBox
Saved first output for: TreasureBox


In [47]:
# Year 11/12 + Hard
gemini("year_11_12","hard")

Generating response for: BeaverDatabase
Saved first output for: BeaverDatabase
Wrong answer detected for: BeaverDatabase
Model answered: Option 1
Correct answer: Option 3
Re-prompting...
Saved retry output for: BeaverDatabase
Generating response for: BeaverGames
Saved first output for: BeaverGames
Generating response for: Ordering2
Saved first output for: Ordering2
Generating response for: SeashellsAndPebbles
Saved first output for: SeashellsAndPebbles
Wrong answer detected for: SeashellsAndPebbles
Model answered: I cannot definitively determine the numbered empty hole that ensures Anushka's victory based on the provided game rules and current board state. My analysis shows that no single move by Anushka forces Bruno to lose on his immediate next turn, as Bruno always has at least one 'safe' empty hole to place his pebble in. However, if forced to choose the most strategically impactful move, it would involve restricting Bruno's future options in a nuanced way that is not explicitly co

**Chat GPT 4.1** 

In [56]:
# Function for gpt
client = OpenAI(api_key = gpt_key)

def gpt(age, dif):
    model_name = "gpt-4o"  # Change to your target GPT-4.1 variant

    task_path = tasks[age][dif]

    # Define the function schema for structured output
    function_schema = {
        "name": "structured_output",
        "description": "Outputs the explanation and final answer.",
        "parameters": {
            "type": "object",
            "properties": {
                "explanation": {
                    "type": "string",
                    "description": "Explanation of the reasoning process"
                },
                "answer": {
                    "type": "string",
                    "description": "Final answer"
                }
            },
            "required": ["explanation", "answer"]
        }
    }

    for task in task_path:
        task_name = os.path.splitext(os.path.basename(task))[0]
        with open(task, "r", encoding="utf-8") as file:
            prompt = file.read()

        print(f"Generating response for: {task_name}")

        full_prompt = (
            "Consider the following task. Please think about it carefully and provide:\n"
            "1. 'explanation': How you solved the task.\n"
            "2. 'answer': Only the final answer (e.g. 'option 2', 'block 3', etc. unless specified differently). Do not add anything else.\n\n"
            f"{prompt}"
        )

        # First attempt
        start_time = time.time()
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": full_prompt}],
            functions=[function_schema],
            function_call={"name": "structured_output"}
        )
        end_time = time.time()
        response_time = round(end_time - start_time, 3)

        args = json.loads(response.choices[0].message.function_call.arguments)
        explanation = args.get("explanation", "").strip()
        predicted = args.get("answer", "").strip()

        correct_answer = answers[age][dif][task_name].strip()
        is_correct = predicted.lower() == correct_answer.lower()

        output_data = {
            "task_name": task_name,
            "prompt": prompt,
            "explanation": explanation,
            "answer": predicted,
            "model": model_name,
            "timestamp": datetime.now().isoformat(),
            "response_time_sec": response_time,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "attempt": "first"
        }

        gpt_output_path = output_path / f"{age}" / f"{dif}" / "gpt"
        gpt_output_path.mkdir(parents=True, exist_ok=True)
        with open(gpt_output_path / f"{task_name}_GPT_4o.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4, ensure_ascii=False)

        print(f"Saved first output for: {task_name}")

        # Retry if incorrect
        if not is_correct:
            print(f"Wrong answer detected for: {task_name}")
            print(f"Model answered: {predicted}")
            print(f"Correct answer: {correct_answer}")
            print("Re-prompting...")

            retry_prompt = (
                f"You previously answered incorrectly.\n"
                f"Your explanation: {explanation}\n"
                f"Your answer: {predicted}\n\n"
                f"Please rethink the problem and try again.\n\n"
                f"{prompt}"
            )

            start_time = time.time()
            retry_response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": retry_prompt}],
                functions=[function_schema],
                function_call={"name": "structured_output"}
            )
            end_time = time.time()
            retry_time = round(end_time - start_time, 3)

            retry_args = json.loads(retry_response.choices[0].message.function_call.arguments)
            retry_expl = retry_args.get("explanation", "").strip()
            retry_ans = retry_args.get("answer", "").strip()

            retry_data = {
                "task_name": task_name,
                "prompt": prompt,
                "previous_answer": predicted,
                "explanation": retry_expl,
                "answer": retry_ans,
                "model": model_name,
                "timestamp": datetime.now().isoformat(),
                "response_time_sec": retry_time,
                "attempt": "retry"
            }

            with open(gpt_output_path / f"{task_name}_GPT_4o_retry.json", "w", encoding="utf-8") as f:
                json.dump(retry_data, f, indent=4, ensure_ascii=False)

            print(f"Saved retry output for: {task_name}")

**Year 7/8**

In [57]:
# Year 7/8 + Easy
gpt("year_7_8","easy")

Generating response for: BeckyBee
Saved first output for: BeckyBee
Wrong answer detected for: BeckyBee
Model answered: 5-15-10-20-9-5-5
Correct answer: 5-10-15-7-10-5-5-10-5
Re-prompting...
Saved retry output for: BeckyBee
Generating response for: FlowerGarden
Saved first output for: FlowerGarden
Generating response for: StoneFactory
Saved first output for: StoneFactory
Generating response for: Strawberries
Saved first output for: Strawberries
Wrong answer detected for: Strawberries
Model answered: 22
Correct answer: 23
Re-prompting...
Saved retry output for: Strawberries
Generating response for: TheGift
Saved first output for: TheGift


In [58]:
# Year 7/8 + Medium
gpt("year_7_8","medium")

Generating response for: ClassroomSeating
Saved first output for: ClassroomSeating
Wrong answer detected for: ClassroomSeating
Model answered: None of the statements can be proven to be false without additional information or assumptions contradicting logical implications.
Correct answer: Statement 3
Re-prompting...
Saved retry output for: ClassroomSeating
Generating response for: ColourTheFrog!
Saved first output for: ColourTheFrog!
Wrong answer detected for: ColourTheFrog!
Model answered: Block 3
Correct answer: Block 4
Re-prompting...
Saved retry output for: ColourTheFrog!
Generating response for: OhridPearls
Saved first output for: OhridPearls
Generating response for: Tic-Tac-Toe
Saved first output for: Tic-Tac-Toe
Wrong answer detected for: Tic-Tac-Toe
Model answered: Option 2
Correct answer: Option 3
Re-prompting...
Saved retry output for: Tic-Tac-Toe
Generating response for: WheatStorage
Saved first output for: WheatStorage
Wrong answer detected for: WheatStorage
Model answered:

In [59]:
# Year 7/8 + Hard
gpt("year_7_8","hard")

Generating response for: ConnectionOfIslands
Saved first output for: ConnectionOfIslands
Wrong answer detected for: ConnectionOfIslands
Model answered: 43
Correct answer: 44
Re-prompting...
Saved retry output for: ConnectionOfIslands
Generating response for: FavouriteMovie
Saved first output for: FavouriteMovie
Wrong answer detected for: FavouriteMovie
Model answered: Grace, Niklaus, Rozsa
Correct answer: Niklaus and Rozsa
Re-prompting...
Saved retry output for: FavouriteMovie
Generating response for: Mysteria
Saved first output for: Mysteria
Wrong answer detected for: Mysteria
Model answered: Option A
Correct answer: Option B
Re-prompting...
Saved retry output for: Mysteria
Generating response for: Ordering1
Saved first output for: Ordering1
Generating response for: UndergroundTrainNetwork
Saved first output for: UndergroundTrainNetwork
Wrong answer detected for: UndergroundTrainNetwork
Model answered: Station C
Correct answer: Station B
Re-prompting...
Saved retry output for: Undergr

**Year 9/10**

In [60]:
# Year 9/10 + Easy
gpt("year_9_10","easy")

Generating response for: Cipher8
Saved first output for: Cipher8
Wrong answer detected for: Cipher8
Model answered: 62-11-26-22-53
Correct answer: 72-11-62-32-43
Re-prompting...
Saved retry output for: Cipher8
Generating response for: InLove
Saved first output for: InLove
Wrong answer detected for: InLove
Model answered: James
Correct answer: Robert
Re-prompting...
Saved retry output for: InLove
Generating response for: MarysNeighbours
Saved first output for: MarysNeighbours
Wrong answer detected for: MarysNeighbours
Model answered: House 5
Correct answer: House 4
Re-prompting...
Saved retry output for: MarysNeighbours
Generating response for: NutsAndBolts
Saved first output for: NutsAndBolts
Wrong answer detected for: NutsAndBolts
Model answered: None of the given options avoid problems for Benoit across the sequence.
Correct answer: Option 3
Re-prompting...
Saved retry output for: NutsAndBolts
Generating response for: RugWeaving
Saved first output for: RugWeaving
Wrong answer detecte

In [61]:
# Year 9/10 + Medium
gpt("year_9_10","medium")

Generating response for: BeaverAI
Saved first output for: BeaverAI
Wrong answer detected for: BeaverAI
Model answered: Option 3: not beaver, beaver
Correct answer: Option 3
Re-prompting...
Saved retry output for: BeaverAI
Generating response for: HangarCarousel
Saved first output for: HangarCarousel
Wrong answer detected for: HangarCarousel
Model answered: 4-1-5-2-6-3
Correct answer: 4-1-3-6-2-5 or 4-1-5-2-6-3
Re-prompting...
Saved retry output for: HangarCarousel
Generating response for: OverlappingVillages
Saved first output for: OverlappingVillages
Wrong answer detected for: OverlappingVillages
Model answered: 8
Correct answer: 5
Re-prompting...
Saved retry output for: OverlappingVillages
Generating response for: SuperSecuritySystem
Saved first output for: SuperSecuritySystem
Wrong answer detected for: SuperSecuritySystem
Model answered: Detectors placed on positions: (1,1), (5,1), (1,5), (5,5), and (3,3)
Correct answer: NA
Re-prompting...
Saved retry output for: SuperSecuritySystem

In [62]:
# Year 9/10 + Medium
gpt("year_9_10","hard")

Generating response for: CollectingStones
Saved first output for: CollectingStones
Wrong answer detected for: CollectingStones
Model answered: 16 beavers
Correct answer: 14 beavers
Re-prompting...
Saved retry output for: CollectingStones
Generating response for: FavouriteGem
Saved first output for: FavouriteGem
Wrong answer detected for: FavouriteGem
Model answered: 11
Correct answer: 10
Re-prompting...
Saved retry output for: FavouriteGem
Generating response for: FourTiles
Saved first output for: FourTiles
Wrong answer detected for: FourTiles
Model answered: Tile 5
Correct answer: Tile 3
Re-prompting...
Saved retry output for: FourTiles
Generating response for: Maze
Saved first output for: Maze
Wrong answer detected for: Maze
Model answered: 16
Correct answer: 18
Re-prompting...
Saved retry output for: Maze
Generating response for: TreeFarming
Saved first output for: TreeFarming
Wrong answer detected for: TreeFarming
Model answered: Position 5th row [3], Position 9th row [1]
Correct a

**Year 11/12**

In [63]:
# Year 11/12 + Easy
gpt("year_11_12","easy")

Generating response for: ColourfulCandles
Saved first output for: ColourfulCandles
Generating response for: ListenAndWalk
Saved first output for: ListenAndWalk
Wrong answer detected for: ListenAndWalk
Model answered: F
Correct answer: B
Re-prompting...
Saved retry output for: ListenAndWalk
Generating response for: Lists
Saved first output for: Lists
Generating response for: MovieNight
Saved first output for: MovieNight
Wrong answer detected for: MovieNight
Model answered: 0-Sara, 1-Joud, 2-Mo, 3-Han, 4-Emy, 5-Jon, 6-Mary
Correct answer: 0-Mo, 1-Joud, 2-Sara, 3-Han, 4-Emy, 5-Jon, 6-Mary
Re-prompting...
Saved retry output for: MovieNight
Generating response for: Words
Saved first output for: Words
Wrong answer detected for: Words
Model answered: 14
Correct answer: 9
Re-prompting...
Saved retry output for: Words


In [64]:
# Year 11/12 + Medium
gpt("year_11_12","medium")

Generating response for: BeaverDam
Saved first output for: BeaverDam
Wrong answer detected for: BeaverDam
Model answered: Option 2
Correct answer: Option 4
Re-prompting...
Saved retry output for: BeaverDam
Generating response for: JumpingGame
Saved first output for: JumpingGame
Wrong answer detected for: JumpingGame
Model answered: [Verunka,OXOXOXOXOXOXOXO,Coin]
Correct answer: [Verunka,X][X][O][X][X][O][O][X][X][O][O][X][X][O][O][Coin]
Re-prompting...
Saved retry output for: JumpingGame
Generating response for: Packing
Saved first output for: Packing
Wrong answer detected for: Packing
Model answered: 2 gaps
Correct answer: NA
Re-prompting...
Saved retry output for: Packing
Generating response for: TreasureBox
Saved first output for: TreasureBox


In [65]:
# Year 11/12 + Hard
gpt("year_11_12","hard")

Generating response for: BeaverDatabase
Saved first output for: BeaverDatabase
Wrong answer detected for: BeaverDatabase
Model answered: Option 1
Correct answer: Option 3
Re-prompting...
Saved retry output for: BeaverDatabase
Generating response for: BeaverGames
Saved first output for: BeaverGames
Wrong answer detected for: BeaverGames
Model answered: C-F-G
Correct answer: D-F-G
Re-prompting...
Saved retry output for: BeaverGames
Generating response for: Ordering2
Saved first output for: Ordering2
Generating response for: SeashellsAndPebbles
Saved first output for: SeashellsAndPebbles
Wrong answer detected for: SeashellsAndPebbles
Model answered: Hole 11
Correct answer: Hole 7
Re-prompting...
Saved retry output for: SeashellsAndPebbles
Generating response for: Virus
Saved first output for: Virus
Wrong answer detected for: Virus
Model answered: 8
Correct answer: 6
Re-prompting...
Saved retry output for: Virus
